# CASE WHEN Transformations

## Overview
`CASE WHEN` is the SQL equivalent of an if-else statement. It is the primary tool for recoding, binning, flagging, and mapping values in a SELECT clause.

**Two forms:**

```sql
-- Searched CASE (most flexible — evaluates arbitrary conditions)
CASE
    WHEN condition1 THEN result1
    WHEN condition2 THEN result2
    ELSE default_result         -- optional; returns NULL if omitted and no match
END

-- Simple CASE (equality comparison against one expression)
CASE expression
    WHEN value1 THEN result1
    WHEN value2 THEN result2
    ELSE default_result
END
```

**Key behaviours:**
- Conditions evaluated in order — the first TRUE condition wins
- If no condition matches and no ELSE, returns NULL
- Can be nested inside aggregates, window functions, ORDER BY, WHERE (via subquery)
- Cannot be used in GROUP BY directly in some databases — use a subquery/CTE

**Common patterns:**
- Recoding free-text to controlled vocabulary
- Binning continuous values into categories
- Creating indicator/flag columns
- Handling NULLs with explicit labels
- Multi-condition scoring

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE intake_raw (record_id INTEGER PRIMARY KEY, patient_ref TEXT, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, phone TEXT, email TEXT, cost_str TEXT, intake_date TEXT, notes TEXT);
CREATE TABLE lab_raw (lab_id INTEGER PRIMARY KEY, patient_ref TEXT, test_code TEXT, result_txt TEXT, collected TEXT, status TEXT);
CREATE TABLE provider_raw (prov_id INTEGER PRIMARY KEY, name_raw TEXT, dept_code TEXT, start_dt TEXT, salary TEXT);
INSERT INTO intake_raw VALUES
  (1,'P-001','aroha','Ngata','1985-03-12','F','NB','506-555-0101','aroha@mail.com','$3,200.00','2023-01-15','Referred by GP'),
  (2,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (3,'P-003','FATIMA','AL-RASHID','1990-07-22','female','Ontario','416-555-0303',NULL,'120.00','2023-03-05','Annual checkup'),
  (4,'P-004','James','MacLeod','Jan 30 1955','M','NB',NULL,'james@mail.com','$5,500','18-03-2023','Surgery follow-up'),
  (5,'P-005','sofia','Petrov','2001-09-15','F','BC','604-555-0505','sofia@mail.com','$95.00','2023-04-02',NULL),
  (6,'P-006','Noah','Williams','08/06/1968','m','AB ','780-555-0606','noah@mail.com','780','11/05/2023',NULL),
  (7,'P-007','Mei','Zhang','1995-04-17','F','ON','416-555-0707',NULL,'$2,100.00','22-06-2023','Follow-up required'),
  (8,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (9,'P-008','ethan','tremblay','01/12/1980',NULL,'QC','514-555-0808','ethan@mail.com','80.00','14-07-2023',NULL),
  (10,'P-009','Priya','Nair','1977-08-25','F','bc',NULL,'priya@mail.com','$1,750.00','01/10/2023',NULL),
  (11,'P-010','Marcus','Okafor','1993-05-19','M','ON','647-555-1010',NULL,'2200','03-11-2023',NULL),
  (12,'P-011','Diana','Flores','14/02/2000','female','NB','506-555-1111','diana@mail.com',NULL,'2023-12-01',NULL),
  (13,NULL,'Unknown',NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,'Incomplete record'),
  (14,'','Test','Record','2023-01-01','X','XX','000-000-0000','test@test.com','-1','2023-01-01','Test entry');
INSERT INTO lab_raw VALUES
  (1,'P-001','HBA1C','7.2 %','2023-03-10','Active'),
  (2,'P-002','HBA1C','9.1%','2023-03-15','active'),
  (3,'P-003','CREAT','88.5umol/L','2023-04-01','ACTIVE'),
  (4,'P-004','CREAT','145 umol/L','2023-04-12','Active'),
  (5,'P-005','HBA1C','5.8 %','2023-05-05',''),
  (6,'P-006','SODIUM','138mmol/L','2023-05-20','Active'),
  (7,'P-007','SODIUM','151 mmol/L','2023-06-01',NULL),
  (8,'P-001','CREAT','72.3 umol/L','2023-06-18','Active'),
  (9,'P-008','HBA1C','6.4%','2023-07-02','Active'),
  (10,'P-009','CREAT','210umol/L','2023-07-15','active');
INSERT INTO provider_raw VALUES
  (1,'MacNeil, Sarah MD','CARD','2018-01-15','$95,000'),
  (2,'Dr. James Wong','ONCO','2019-03-01','88000'),
  (3,'Fatima Osei M.D.','FAM','2017-06-01','$82,500.00'),
  (4,'Larson, Ethan','ORTH','2020-09-15','91000.00'),
  (5,'Sharma, Priya MD','EMRG','2016-11-01','$78,000'),
  (6,'Lucas Petit','RAD','2021-02-28',NULL);
""")
conn.commit()
print("Schema ready.")

Schema ready.


---
## Recoding — mapping messy values to a controlled vocabulary

In [2]:
print("=== Recode sex and province to canonical values ===")
q = """
SELECT  record_id,
        sex,
        CASE UPPER(TRIM(COALESCE(sex,'')))
            WHEN 'M'      THEN 'Male'
            WHEN 'MALE'   THEN 'Male'
            WHEN 'F'      THEN 'Female'
            WHEN 'FEMALE' THEN 'Female'
            WHEN ''       THEN NULL
            ELSE 'Unknown'
        END  AS sex_coded,
        province,
        CASE UPPER(TRIM(COALESCE(province,'')))
            WHEN 'NB'      THEN 'NB'
            WHEN 'NS'      THEN 'NS'
            WHEN 'ON'      THEN 'ON'
            WHEN 'ONTARIO' THEN 'ON'
            WHEN 'BC'      THEN 'BC'
            WHEN 'AB'      THEN 'AB'
            WHEN 'QC'      THEN 'QC'
            WHEN ''       THEN NULL
            ELSE 'Invalid'
        END  AS province_coded
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

=== Recode sex and province to canonical values ===
 record_id    sex sex_coded province province_coded
         1      F    Female       NB             NB
         2   Male      Male       NS             NS
         3 female    Female  Ontario             ON
         4      M      Male       NB             NB
         5      F    Female       BC             BC
         6      m      Male      AB              AB
         7      F    Female       ON             ON
         8   Male      Male       NS             NS
         9   None      None       QC             QC
        10      F    Female       bc             BC
        11      M      Male       ON             ON
        12 female    Female       NB             NB
        13   None      None     None           None
        14      X   Unknown       XX        Invalid


---
## Binning — continuous values into categories

In [3]:
# Bin numeric lab results into clinical ranges
print("=== Bin HbA1c into clinical categories ===")
q = """
SELECT  lab_id,
        patient_ref,
        test_code,
        result_txt,
        CAST(TRIM(REPLACE(REPLACE(result_txt,'%',''),' ','')) AS REAL) AS result_num,
        CASE
            WHEN test_code != 'HBA1C' THEN 'N/A'
            WHEN CAST(TRIM(REPLACE(REPLACE(result_txt,'%',''),' ','')) AS REAL) < 5.7
                THEN 'Normal (<5.7%)'
            WHEN CAST(TRIM(REPLACE(REPLACE(result_txt,'%',''),' ','')) AS REAL) < 6.5
                THEN 'Pre-diabetic (5.7-6.4%)'
            ELSE 'Diabetic (>=6.5%)'
        END  AS hba1c_category
FROM    lab_raw
WHERE   test_code = 'HBA1C'
ORDER BY result_num
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Bin cost into financial tiers
print()
print("=== Bin encounter costs into tiers ===")
q2 = """
SELECT  record_id,
        cost_str,
        CAST(REPLACE(REPLACE(COALESCE(cost_str,'0'),'$',''),',','') AS REAL) AS cost_num,
        CASE
            WHEN cost_str IS NULL THEN 'Missing'
            WHEN CAST(REPLACE(REPLACE(cost_str,'$',''),',','') AS REAL) <= 0
                THEN 'Zero/Invalid'
            WHEN CAST(REPLACE(REPLACE(cost_str,'$',''),',','') AS REAL) < 200
                THEN 'Low (<$200)'
            WHEN CAST(REPLACE(REPLACE(cost_str,'$',''),',','') AS REAL) < 2000
                THEN 'Medium ($200-$2k)'
            ELSE 'High (>=$2k)'
        END  AS cost_tier
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q2, conn).to_string(index=False))

=== Bin HbA1c into clinical categories ===
 lab_id patient_ref test_code result_txt  result_num          hba1c_category
      5       P-005     HBA1C      5.8 %         5.8 Pre-diabetic (5.7-6.4%)
      9       P-008     HBA1C       6.4%         6.4 Pre-diabetic (5.7-6.4%)
      1       P-001     HBA1C      7.2 %         7.2       Diabetic (>=6.5%)
      2       P-002     HBA1C       9.1%         9.1       Diabetic (>=6.5%)

=== Bin encounter costs into tiers ===
 record_id  cost_str  cost_num         cost_tier
         1 $3,200.00    3200.0      High (>=$2k)
         2      1850    1850.0 Medium ($200-$2k)
         3    120.00     120.0       Low (<$200)
         4    $5,500    5500.0      High (>=$2k)
         5    $95.00      95.0       Low (<$200)
         6       780     780.0 Medium ($200-$2k)
         7 $2,100.00    2100.0      High (>=$2k)
         8      1850    1850.0 Medium ($200-$2k)
         9     80.00      80.0       Low (<$200)
        10 $1,750.00    1750.0 Medium ($20

---
## Flag creation — indicator columns for downstream filtering

In [4]:
print("=== Create quality flags for each intake record ===")
q = """
SELECT  record_id,
        patient_ref,
        CASE WHEN patient_ref IS NULL OR patient_ref = '' THEN 1 ELSE 0 END AS flag_no_ref,
        CASE WHEN first_name IS NULL OR last_name IS NULL   THEN 1 ELSE 0 END AS flag_incomplete_name,
        CASE WHEN dob IS NULL                               THEN 1 ELSE 0 END AS flag_no_dob,
        CASE WHEN sex NOT IN ('M','F') OR sex IS NULL      THEN 1 ELSE 0 END AS flag_bad_sex,
        CASE WHEN phone IS NULL AND email IS NULL           THEN 1 ELSE 0 END AS flag_no_contact,
        CASE WHEN cost_str IS NULL OR
                  CAST(REPLACE(REPLACE(cost_str,'$',''),',','') AS REAL) < 0
             THEN 1 ELSE 0 END  AS flag_bad_cost
FROM    intake_raw
ORDER BY record_id
"""
df = pd.read_sql(q, conn)
print(df.to_string(index=False))
print()
df_flags = df[['flag_no_ref','flag_incomplete_name','flag_no_dob','flag_bad_sex','flag_no_contact','flag_bad_cost']]
print("Flag totals:")
print(df_flags.sum().to_string())

=== Create quality flags for each intake record ===
 record_id patient_ref  flag_no_ref  flag_incomplete_name  flag_no_dob  flag_bad_sex  flag_no_contact  flag_bad_cost
         1       P-001            0                     0            0             0                0              0
         2       P-002            0                     0            0             1                0              0
         3       P-003            0                     0            0             1                0              0
         4       P-004            0                     0            0             0                0              0
         5       P-005            0                     0            0             0                0              0
         6       P-006            0                     0            0             1                0              0
         7       P-007            0                     0            0             0                0              0
         8  

---
## CASE WHEN in ORDER BY and with aggregates

In [5]:
# Custom sort order using CASE WHEN in ORDER BY
print("=== Custom sort order with CASE WHEN in ORDER BY ===")
q = """
SELECT  record_id, province,
        CASE UPPER(TRIM(COALESCE(province,'')))
            WHEN 'ON' THEN 'ON'
            WHEN 'ONTARIO' THEN 'ON'
            WHEN 'BC' THEN 'BC'
            WHEN 'AB' THEN 'AB'
            WHEN 'NB' THEN 'NB'
            WHEN 'NS' THEN 'NS'
            WHEN 'QC' THEN 'QC'
            ELSE 'Other/Invalid'
        END AS province_clean
FROM    intake_raw
ORDER BY
    CASE UPPER(TRIM(COALESCE(province,'')))
        WHEN 'ON' THEN 1 WHEN 'ONTARIO' THEN 1
        WHEN 'BC' THEN 2
        WHEN 'AB' THEN 3
        WHEN 'QC' THEN 4
        WHEN 'NB' THEN 5 WHEN 'NS' THEN 6
        ELSE 99
    END, record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

# CASE WHEN inside COUNT for multi-category summary
print()
print("=== Multi-flag summary using conditional COUNT ===")
q2 = """
SELECT
    COUNT(*) AS total_records,
    COUNT(CASE WHEN patient_ref IS NULL OR patient_ref='' THEN 1 END) AS missing_ref,
    COUNT(CASE WHEN sex NOT IN ('M','F') OR sex IS NULL THEN 1 END) AS bad_sex,
    COUNT(CASE WHEN phone IS NULL AND email IS NULL THEN 1 END) AS no_contact,
    COUNT(CASE WHEN dob IS NULL THEN 1 END) AS missing_dob,
    ROUND(100.0 * COUNT(CASE WHEN patient_ref IS NULL OR patient_ref='' THEN 1 END) / COUNT(*), 1) AS pct_missing_ref
FROM intake_raw
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()

=== Custom sort order with CASE WHEN in ORDER BY ===
 record_id province province_clean
         3  Ontario             ON
         7       ON             ON
        11       ON             ON
         5       BC             BC
        10       bc             BC
         6      AB              AB
         9       QC             QC
         1       NB             NB
         4       NB             NB
        12       NB             NB
         2       NS             NS
         8       NS             NS
        13     None  Other/Invalid
        14       XX  Other/Invalid

=== Multi-flag summary using conditional COUNT ===
 total_records  missing_ref  bad_sex  no_contact  missing_dob  pct_missing_ref
            14            2        8           1            1             14.3


---
## Common Pitfalls

**1. ELSE NULL is implicit but easy to forget**
`CASE WHEN sex = 'F' THEN 'Female' WHEN sex = 'M' THEN 'Male' END` — rows where sex is NULL, 'Male', 'female', etc. all return NULL silently. Add `ELSE 'Unknown'` or `ELSE NULL -- documented` to make the fallback explicit and intentional.

**2. Order of WHEN conditions matters — first match wins**
`CASE WHEN cost > 0 THEN 'Positive' WHEN cost > 1000 THEN 'High'` — the second WHEN is unreachable because every cost > 1000 also satisfies cost > 0. Order from most specific to least specific: check `cost > 1000` before `cost > 0`.

**3. CASE WHEN on a column doesn't handle NULLs automatically**
`CASE WHEN sex = 'X' THEN 'Invalid'` — when sex IS NULL, the condition `NULL = 'X'` is NULL (not FALSE), so the WHEN is skipped. NULLs fall through to ELSE. If NULLs need special treatment, add an explicit `WHEN sex IS NULL THEN 'Missing'` before other conditions.

**4. Repeating complex expressions inside CASE for both WHEN and the recoded value**
`CASE WHEN CAST(REPLACE(cost_str,'$','') AS REAL) > 1000 THEN CAST(REPLACE(cost_str,'$','') AS REAL)` duplicates the expression. Use a CTE or derived table to compute the cleaned value once, then apply CASE WHEN to the clean column.

**5. Using CASE WHEN in GROUP BY directly**
`GROUP BY CASE WHEN cost > 1000 THEN 'High' ELSE 'Low' END` works in PostgreSQL but may cause issues with column aliases in some databases. The safest pattern is to compute the CASE WHEN in a CTE and GROUP BY the alias in the outer query.

**6. Simple CASE form doesn't handle NULL equality**
`CASE sex WHEN NULL THEN 'Missing'` never matches because `sex = NULL` is NULL. Use the searched form: `CASE WHEN sex IS NULL THEN 'Missing'`.


---
*sql_methods_library - Samantha McGarrigle*